In [ ]:
!pip install mtcnn tensorflow opencv-python scikit-learn matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 38.0 MB/s eta 0:00:00


In [ ]:
import os
import cv2
import numpy as np
import shutil
from sklearn.model_selection import train_test_split
from mtcnn.mtcnn import MTCNN

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# --- KONFIGURASI ---
SOURCE_VIDEO_DIR = "/content/drive/MyDrive/datasetbaru" # Folder input video Anda
BASE_FRAME_DIR = "/content/drive/MyDrive/datasetbaru/frame"   # Folder output untuk WAJAH yang di-crop
CATEGORIES = ["asli", "deepfake"]

# Pengaturan Ekstraksi & Ukuran
FRAMES_PER_VIDEO = 10
IMG_SIZE = (224, 224)  # Target ukuran untuk EfficientNetB0
FACE_PADDING = 0

# Pengaturan Pembagian Data
TRAIN_RATIO = 0.7
VALIDATION_RATIO = 0.2
TEST_RATIO = 0.1

In [ ]:
print("Menginisialisasi detektor wajah MTCNN...")
try:
    detector = MTCNN()
except Exception as e:
    print(f"Error saat inisialisasi MTCNN: {e}")
    print("Pastikan Anda telah menginstal tensorflow (pip install tensorflow)")
    exit()

def extract_faces(video_path, dest_folder, video_filename_base):
    """
    Membaca video, mendeteksi wajah, meng-crop, me-resize ke 224x224,
    dan menyimpannya sebagai file gambar.
    """
    if not os.path.exists(dest_folder):
        os.makedirs(dest_folder)

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error: Tidak bisa membuka video {video_path}")
        return

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames == 0:
        cap.release()
        return

    num_frames_to_extract = min(FRAMES_PER_VIDEO, total_frames)
    frame_indices = np.linspace(0, total_frames - 1, num_frames_to_extract, dtype=int)

    count = 0
    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()

        if ret:
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            try:
                faces = detector.detect_faces(frame_rgb)
            except Exception:
                continue # Lewati frame jika MTCNN error

            if len(faces) == 1:
                x, y, w, h = faces[0]['box']
                x1, y1 = max(0, x - FACE_PADDING), max(0, y - FACE_PADDING)
                x2, y2 = min(frame.shape[1], x + w + FACE_PADDING), min(frame.shape[0], y + h + FACE_PADDING)
                face_crop = frame[y1:y2, x1:x2]

                if face_crop.size > 0:
                    face_resized = cv2.resize(face_crop, IMG_SIZE)
                    output_filename = os.path.join(dest_folder, f"{video_filename_base}_face_{count+1}.jpg")
                    cv2.imwrite(output_filename, face_resized)
                    count += 1
    cap.release()

Menginisialisasi detektor wajah MTCNN...


In [ ]:
def main():
    print("Memulai proses ekstraksi WAJAH...")
    if os.path.exists(BASE_FRAME_DIR):
        shutil.rmtree(BASE_FRAME_DIR)

    for split in ['train', 'validation', 'test']:
        for category in CATEGORIES:
            os.makedirs(os.path.join(BASE_FRAME_DIR, split, category), exist_ok=True)

    for category in CATEGORIES:
        print(f"\nMemproses kategori: {category}")
        category_video_path = os.path.join(SOURCE_VIDEO_DIR, category)
        try:
            video_files = [f for f in os.listdir(category_video_path) if f.endswith(('.mp4', '.avi', '.mov'))]
        except FileNotFoundError:
            print(f"Error: Folder {category_video_path} tidak ditemukan.")
            continue

        if not video_files: continue

        videos_train, videos_val_test = train_test_split(video_files, test_size=(1 - TRAIN_RATIO), random_state=42)
        relative_test_ratio = TEST_RATIO / (VALIDATION_RATIO + TEST_RATIO)

        if not videos_val_test:
            videos_val, videos_test = [], []
        else:
            videos_val, videos_test = train_test_split(videos_val_test, test_size=relative_test_ratio, random_state=42)

        print(f"  {category}: {len(videos_train)} train, {len(videos_val)} validation, {len(videos_test)} test videos.")

        video_splits = {'train': videos_train, 'validation': videos_val, 'test': videos_test}

        for split_name, video_list in video_splits.items():
            dest_path_base = os.path.join(BASE_FRAME_DIR, split_name, category)
            for video_file in video_list:
                source_video_path = os.path.join(category_video_path, video_file)
                video_filename_base = os.path.splitext(video_file)[0]
                extract_faces(source_video_path, dest_path_base, video_filename_base)

    print(f"\nEkstraksi wajah selesai. Data siap di folder: {BASE_FRAME_DIR}")

if __name__ == "__main__":
    main()

Memulai proses ekstraksi WAJAH...

Memproses kategori: asli
  asli: 104 train, 30 validation, 16 test videos.

Memproses kategori: deepfake
  deepfake: 118 train, 34 validation, 18 test videos.

Ekstraksi wajah selesai. Data siap di folder: /content/drive/MyDrive/datasetbaru/frame


In [ ]:
import os

def count_files_in_folder(folder_path):
    count = 0
    if not os.path.exists(folder_path):
        # print(f"Folder tidak ditemukan: {folder_path}") # Uncomment for debugging if needed
        return 0
    for root, _, files in os.walk(folder_path):
        for file in files:
            count += 1
    return count

# Define splits based on the previous configuration
splits = {'train': TRAIN_RATIO, 'validation': VALIDATION_RATIO, 'test': TEST_RATIO}

print("Jumlah file per kategori dalam setiap folder:")
for split_name in splits.keys():
    print(f"\n--- Folder: {split_name} ---")
    split_dir = os.path.join(BASE_FRAME_DIR, split_name)
    if not os.path.exists(split_dir):
        print(f"  Folder {split_dir} tidak ditemukan.")
        continue

    total_files_in_split = 0
    for category in CATEGORIES:
        category_dir = os.path.join(split_dir, category)
        files_count = count_files_in_folder(category_dir)
        print(f"  Kategori '{category}': {files_count} file")
        total_files_in_split += files_count
    print(f"  Total file di '{split_name}': {total_files_in_split}")

# Also print overall totals for quick summary (optional, but good for context)
print("\n--- Total Keseluruhan ---")
total_train_files = count_files_in_folder(os.path.join(BASE_FRAME_DIR, 'train'))
total_validation_files = count_files_in_folder(os.path.join(BASE_FRAME_DIR, 'validation'))
total_test_files = count_files_in_folder(os.path.join(BASE_FRAME_DIR, 'test'))

print(f"Total file di semua kategori dalam 'train': {total_train_files}")
print(f"Total file di semua kategori dalam 'validation': {total_validation_files}")
print(f"Total file di semua kategori dalam 'test': {total_test_files}")


Jumlah file per kategori dalam setiap folder:

--- Folder: train ---
  Kategori 'asli': 1254 file
  Kategori 'deepfake': 1136 file
  Total file di 'train': 2390

--- Folder: validation ---
  Kategori 'asli': 255 file
  Kategori 'deepfake': 337 file
  Total file di 'validation': 592

--- Folder: test ---
  Kategori 'asli': 199 file
  Kategori 'deepfake': 193 file
  Total file di 'test': 392

--- Total Keseluruhan ---
Total file di semua kategori dalam 'train': 2390
Total file di semua kategori dalam 'validation': 692
Total file di semua kategori dalam 'test': 392


In [ ]:
import os

folder_path = '/content/drive/MyDrive/datasetbaru/lainlain/500/test'
nama_baru_base = "wild"
lakukan_rename = True

def batch_rename(path, base_name, execute):
    # Cek apakah folder ada
    if not os.path.exists(path):
        print(f"❌ Error: Folder tidak ditemukan di: {path}")
        return

    # Ambil list file dan urutkan agar rapi
    files = sorted(os.listdir(path))

    count = 0
    print(f"{'='*20} MULAI PROSES ({'EKSEKUSI' if execute else 'SIMULASI'}) {'='*20}")

    for index, filename in enumerate(files):
        # Dapatkan full path file lama
        old_file_path = os.path.join(path, filename)

        # Pastikan yang di-rename adalah file, bukan folder
        if os.path.isfile(old_file_path):
            # Ambil ekstensi file asli (misal: .jpg, .mp4, .png)
            file_extension = os.path.splitext(filename)[1]

            # Format nama baru: Label_001.ext (padding angka 3 digit)
            new_filename = f"{base_name}_{index+1:03d}{file_extension}"
            new_file_path = os.path.join(path, new_filename)

            # Cek agar tidak me-rename file yang namanya sudah sama (opsional)
            if filename != new_filename:
                if execute:
                    # Lakukan Rename Asli
                    os.rename(old_file_path, new_file_path)
                    print(f"✅ Berhasil: '{filename}'  --->  '{new_filename}'")
                else:
                    # Hanya Print Preview
                    print(f"👁️ Preview: '{filename}'  --->  '{new_filename}'")
                count += 1
            else:
                print(f"⚠️ Skip: '{filename}' (Nama sudah sesuai)")

    print(f"\n{'='*20} SELESAI. Total file diproses: {count} {'='*20}")

# Jalankan Fungsi
batch_rename(folder_path, nama_baru_base, lakukan_rename)

==================== MULAI PROSES (EKSEKUSI) ====================
✅ Berhasil: '0_14.png'  --->  'wild_001.png'
✅ Berhasil: '0_18.png'  --->  'wild_002.png'
✅ Berhasil: '0_29.png'  --->  'wild_003.png'
✅ Berhasil: '0_293.png'  --->  'wild_004.png'
✅ Berhasil: '0_294.png'  --->  'wild_005.png'
✅ Berhasil: '0_323.png'  --->  'wild_006.png'
✅ Berhasil: '0_355.png'  --->  'wild_007.png'
✅ Berhasil: '0_398.png'  --->  'wild_008.png'
✅ Berhasil: '0_40.png'  --->  'wild_009.png'
✅ Berhasil: '0_51.png'  --->  'wild_010.png'
✅ Berhasil: '0_54.png'  --->  'wild_011.png'
✅ Berhasil: '0_6.png'  --->  'wild_012.png'
✅ Berhasil: '0_66.png'  --->  'wild_013.png'
✅ Berhasil: '0_74.png'  --->  'wild_014.png'
✅ Berhasil: '0_83.png'  --->  'wild_015.png'
✅ Berhasil: '0_9.png'  --->  'wild_016.png'
✅ Berhasil: '0_91.png'  --->  'wild_017.png'
✅ Berhasil: '0_93.png'  --->  'wild_018.png'
✅ Berhasil: '1_322.png'  --->  'wild_019.png'
✅ Berhasil: '1_323.png'  --->  'wild_020.png'
✅ Berhasil: '1_325.png'  --->

### Ringkasan Jumlah Frame per Kategori dan Split

In [ ]:
print("Menghitung jumlah frame untuk setiap kategori di setiap split secara otomatis...")

asli_train = count_files_in_folder(os.path.join(BASE_FRAME_DIR, 'train', 'asli'))
deepfake_train = count_files_in_folder(os.path.join(BASE_FRAME_DIR, 'train', 'deepfake'))

asli_validation = count_files_in_folder(os.path.join(BASE_FRAME_DIR, 'validation', 'asli'))
deepfake_validation = count_files_in_folder(os.path.join(BASE_FRAME_DIR, 'validation', 'deepfake'))

asli_test = count_files_in_folder(os.path.join(BASE_FRAME_DIR, 'test', 'asli'))
deepfake_test = count_files_in_folder(os.path.join(BASE_FRAME_DIR, 'test', 'deepfake'))

print(f"\n--- Detail Jumlah Frame ---")
print(f"Train Split:")
print(f"  Kategori 'asli': {asli_train} frames")
print(f"  Kategori 'deepfake': {deepfake_train} frames")
print(f"\nValidation Split:")
print(f"  Kategori 'asli': {asli_validation} frames")
print(f"  Kategori 'deepfake': {deepfake_validation} frames")
print(f"\nTest Split:")
print(f"  Kategori 'asli': {asli_test} frames")
print(f"  Kategori 'deepfake': {deepfake_test} frames")

# Total keseluruhan untuk setiap kategori
total_asli_overall = asli_train + asli_validation + asli_test
total_deepfake_overall = deepfake_train + deepfake_validation + deepfake_test

print(f"\n--- Total Keseluruhan per Kategori ---")
print(f"Total 'asli' di semua split: {total_asli_overall} frames")
print(f"Total 'deepfake' di semua split: {total_deepfake_overall} frames")

Menghitung jumlah frame untuk setiap kategori di setiap split secara otomatis...

--- Detail Jumlah Frame ---
Train Split:
  Kategori 'asli': 1254 frames
  Kategori 'deepfake': 1136 frames

Validation Split:
  Kategori 'asli': 255 frames
  Kategori 'deepfake': 337 frames

Test Split:
  Kategori 'asli': 199 frames
  Kategori 'deepfake': 193 frames

--- Total Keseluruhan per Kategori ---
Total 'asli' di semua split: 1708 frames
Total 'deepfake' di semua split: 1666 frames
